<a href="https://colab.research.google.com/github/shuangquan-li-con/angrist-evans-1998-iv-replication/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install linearmodels

In [8]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

df = pd.read_csv("/content/cleaned_combined_data.csv")

df = df[["SEXK", "SEX2NDK", "KIDCOUNT", "WEEK89M", "HOUR89M", "AGEM", "YEARSCHM"]].copy()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()
df["same_sex"] = (df["SEXK"] == df["SEX2NDK"]).astype(int)
df["more_kids"] = (df["KIDCOUNT"] >= 30).astype(int)
df["worked"] = ((df["WEEK89M"] > 0) | (df["HOUR89M"] > 0)).astype(int)

print("KIDCOUNT value counts after cleaning:")
print(df["KIDCOUNT"].value_counts())
print("\nmore_kids value counts after cleaning:")
print(df["more_kids"].value_counts())

X1 = sm.add_constant(df[["same_sex", "AGEM", "YEARSCHM"]])
y1 = df["more_kids"]

first_stage = sm.OLS(y1, X1).fit()
print("First Stage")
print(first_stage.summary())

df["more_kids_hat"] = first_stage.predict(X1)

X2 = sm.add_constant(df[["more_kids_hat", "AGEM", "YEARSCHM"]])
y2 = df["worked"]

manual_2sls = sm.OLS(y2, X2).fit()
print("Manual 2SLS")
print(manual_2sls.summary())

iv_model = IV2SLS(
    dependent=df["worked"],
    exog=sm.add_constant(df[["AGEM", "YEARSCHM"]]),
    endog=df["more_kids"],
    instruments=df["same_sex"]
).fit()

print("IV2SLS")
print(iv_model.summary)


y_z1 = df.loc[df["same_sex"] == 1, "worked"].mean()
y_z0 = df.loc[df["same_sex"] == 0, "worked"].mean()

d_z1 = df.loc[df["same_sex"] == 1, "more_kids"].mean()
d_z0 = df.loc[df["same_sex"] == 0, "more_kids"].mean()

wald = (y_z1 - y_z0) / (d_z1 - d_z0)

print("Wald Estimator =", wald)


KIDCOUNT value counts after cleaning:
KIDCOUNT
20     24557
30      8496
40      2189
50       486
60       128
70        45
80         8
90         7
100        3
110        2
Name: count, dtype: int64

more_kids value counts after cleaning:
more_kids
0    24557
1    11364
Name: count, dtype: int64
First Stage
                            OLS Regression Results                            
Dep. Variable:              more_kids   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     231.1
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          1.70e-148
Time:                        03:42:18   Log-Likelihood:                -23125.
No. Observations:               35921   AIC:                         4.626e+04
Df Residuals:                   35917   BIC:                         4.629e+04
Df Model:                           3                  